<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/combinatorial_bandit_unsupervised_c2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-view Data Selection Simulation

This notebook contains a simulation for optimal multi-view data selection, focusing on how to dynamically select subsets of data views to maximize reward under a budget constraint.

## 1. Import Libraries

We begin by importing all the necessary Python libraries for numerical operations, data manipulation, plotting, and machine learning utilities.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import scipy.optimize as opt
from scipy.stats import norm
from sklearn.metrics import confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

## 2. Define Simulation Parameters

Instead of using `argparse`, we define the simulation parameters directly as variables here. You can easily modify these values to experiment with different scenarios.

In [2]:
# Simulation Parameters
seed = 42 # Random seed for reproducibility
num_views = 5 # Number of data views
num_rounds = 2000 # Number of simulation rounds (samples)
num_trials = 20 # Number of independent trials to run
budget_ratio = 0.7 # Ratio of the total possible cost that can be spent
output_filename = "output_gmm2_unsu.csv" # Output CSV file name

## 3. Data Generation Function

`generate_data` creates synthetic multi-view data. It uses `sklearn.datasets.make_blobs` to generate clusters, simulating different classes across multiple views (features).

In [3]:
def generate_data(rng,nsamples=1000,nclasses=2,nviews=10,seed=42,snrdb=10):
    sval = 10**(snrdb/20)
    mm = rng.random(size=(nclasses,nviews)) * sval # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1]

## 4. Optimal Static Strategy (`optimal_static`)

This function calculates the optimal reward and cost for a static selection strategy using linear programming. It considers all possible subsets of views and their associated rewards (accuracy) and costs to find the best distribution of view selections under a given budget.

In [4]:
def optimal_static(means,costs,budget,num_rounds):
    nviews = len(costs)
    noise_vars = 1.0 # FIXME: for a single dimension, for now
    mean_diff_sq = 0.5*np.square((means[0,:]- means[1,:])/noise_vars)
    lp_table = np.zeros((2**nviews-1,2)) # (expected reward, costs)
    for comb in range(1,2**nviews):
        sel_idx = np.array([int(bit) for bit in f"{comb:0{nviews}b}"]).astype(bool)
        snrs = np.sum(mean_diff_sq[sel_idx])
        err_rate_calc = norm.sf(np.sqrt(snrs), loc=0,scale=1) # survival function

        cost_sum = np.sum(costs[sel_idx])
        lp_table[comb-1,:] = np.array([1-err_rate_calc,cost_sum])

    budget_ratio = budget / num_rounds
    r_expect = np.expand_dims(lp_table[:,0],axis=0)
    r_cost = np.expand_dims(lp_table[:,1],axis=0)
    simplex_sum = np.ones((1,2**nviews-1))
    x_bound = [(0,None) for _ in range(2**nviews-1)]
    res = opt.linprog(-r_expect,A_ub=r_cost,b_ub=budget_ratio,A_eq=simplex_sum, b_eq=1,bounds=x_bound)
    rt_msg = res.fun
    opt_pi = res.x # this is the optimal static distribution
    opt_reward = num_rounds * r_expect @ opt_pi
    opt_costs = num_rounds * r_cost @ opt_pi

    print(f"Rounds:{num_rounds}, Budget:{budget}, Optimal Reward: {opt_reward.item():.4e}, Optimal_cost: {opt_costs.item():.4e}")
    return {"solver_message":rt_msg,"reward":opt_reward.item(),"cost":opt_costs.item(),"solution":opt_pi}

## 5. Subset Selection Function (`get_subset`)

This function determines the best subset of views to select greedily in each round based on the estimated reward and a dual variable (`lambda_dual`) that balances reward and cost. It aims to maximize the net benefit by considering both accuracy and cost.

In [5]:
def get_subset(subset_diff_vars,costs,lambda_dual,spending_ratio):
    best_reward = -np.inf
    best_subset = None
    nviews = len(costs)
    for v in range(1, 2**nviews):
        subset = np.array([int(bit) for bit in f"{v:0{nviews}b}"]).astype(bool)
        tmp_cost = np.sum(costs[subset])
        # For a 2-component GMM, the argument to the error function is sqrt(sum( (mu_1-mu_2)^2 / (4*sigma^2) ))
        snrs = subset_diff_vars[subset] # Assuming sigma^2 = 1
        tmp_err_rate = norm.sf(0.5*np.sqrt(np.sum(snrs)), loc=0, scale=1)
        tmp_reward = 1-tmp_err_rate - lambda_dual * (tmp_cost - spending_ratio)
        if tmp_reward > best_reward:
            best_reward = tmp_reward
            best_subset = subset
    # greedily choose the maximum
    return best_subset

## 6. Mean and Covariance Calculation Functions

These helper functions assist in estimating the means and covariances of the data from selected subsets, accounting for varying observation counts across different views.

In [6]:
def calc_subset_mean(x_obs,counts,mean_indices):
    n_sum = np.array([sum([counts[idx] for _,idx in dd.items()]) for dd in mean_indices])
    # Avoid division by zero
    return np.divide(x_obs, n_sum, out=np.zeros_like(x_obs), where=n_sum!=0)

def calc_subset_cov(x_cov,counts,cov_indices,utri_idcs_k1, est_subset_means):
    nv = x_cov.shape[0]
    n_sum = np.zeros((nv,nv))
    for v in range(nv):
        n_sum[v,v] = sum([counts[idx] for _,idx in cov_indices[v][v].items()])
        for w in range(v+1,nv):
            n_sum[v,w] = sum([counts[idx] for _,idx in cov_indices[v][w].items()])
    n_sum[utri_idcs_k1[1],utri_idcs_k1[0]] = n_sum[utri_idcs_k1[0],utri_idcs_k1[1]]
    # Calculate E[X*X^T] with safe division
    second_moment = np.divide(x_cov, n_sum, out=np.zeros_like(x_cov), where=n_sum!=0)
    # Cov(X) = E[X*X^T] - E[X]E[X]^T
    return second_moment - np.outer(est_subset_means, est_subset_means)

## 7. Label Matching Function (`label_matching`)

This function handles the label permutation issue common in unsupervised learning. It uses the Hungarian algorithm (via `linear_sum_assignment`) to match predicted labels to true labels, maximizing the accuracy of the clustering or classification.

In [7]:
def label_matching(y_true, y_pred):
    assert(len(y_pred) == len(y_true))
    # 1. Compute confusion matrix
    D = confusion_matrix(y_true, y_pred)
    # 2. Solve the linear sum assignment problem
    # We negate the matrix because the algorithm minimizes cost, and we want to maximize matches
    row_ind, col_ind = linear_sum_assignment(D.max() - D)
    # 3. Create a mapping from predicted to true labels
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    # 4. Apply the mapping to the predictions
    y_pred_mapped = np.array([mapping[label] for label in y_pred])
    return y_pred_mapped, mapping

## 8. Index Mapping Function (`get_indices`)

`get_indices` precomputes dictionaries that map view subset combinations to their corresponding indices. This helps efficiently aggregate counts for mean and covariance estimation across different selected views.

In [8]:
def get_indices(num_views):
    mean_indice =[{} for _ in range(num_views)] # record the subset indices to sum
    cov_indice = [[{} for _ in range(num_views)] for _ in range(num_views)]
    for b in range(1,2**num_views):
        b_offset = b - 1 # to offset the index
        sel_idx = np.array([int(bit) for bit in f"{b:0{num_views}b}"])
        key_str = "".join(sel_idx.astype(str))
        for v in range(num_views):
            if sel_idx[v]:
                mean_indice[v][key_str] = b_offset
            for w in range(num_views):
                if sel_idx[w] and sel_idx[v]:
                        cov_indice[v][w][key_str] = b_offset
    return mean_indice, cov_indice

## 9. Simulation Function (`simulation`)

This is the core simulation logic. It iteratively selects data views, updates estimators, and makes predictions. It uses an Online Mirror Descent (OMD) approach to adapt the selection strategy over time, balancing exploration (estimating parameters) and exploitation (choosing views that yield high reward).

In [13]:
def simulation(num_views,costs,num_rounds,budget,x_data,y_labels,**kwargs):
    # initialize the estimators
    est_counts = np.zeros((2**num_views-1))
    mean_indice, cov_indice = get_indices(num_views)
    est_sample_means = np.zeros((num_views))
    est_sample_cov = np.eye(num_views) # regularized

    remain_budget = budget
    spending_ratio = budget / num_rounds
    acc_reward = 0
    omd_lambda = 0
    lambda_max = 10 # change this when needed
    step_size = 1.0
    noise_var = 1.0
    # optimistic exploration parameter
    xi_fixed = 2.0
    utri_idcs_k1 = np.triu_indices(num_views,k=1)

    # recording purpose
    true_mean_diff = kwargs['means'][0,:] - kwargs['means'][1,:]
    record_sign_eigv = np.zeros((num_rounds))

    # recording
    record_predict = np.zeros((num_rounds))
    for nr in range(num_rounds):
        # we need to sum the counters to get the sample mean
        if nr != 0:
            # add the optimistic means
            est_subset_means = calc_subset_mean(est_sample_means,est_counts,mean_indice)
            est_subset_cov = calc_subset_cov(est_sample_cov,est_counts,cov_indice,utri_idcs_k1, est_subset_means)
            # the difference estimates need to add the per dimension exploration count
            tmp_count = np.array([sum([est_counts[v] for _,v in dd.items()]) for dd in mean_indice])
            # Ensure tmp_count doesn't have zeros to prevent division by zero in sqrt
            tmp_count_safe = np.where(tmp_count == 0, 1e-6, tmp_count) # Replace 0 with a small positive number

            subset_var = np.maximum(np.diag(est_subset_cov)/noise_var - 1, 1e-3) + np.sqrt(xi_fixed * np.log(nr+1)/ tmp_count_safe)
            subset = get_subset(subset_var,costs,omd_lambda,spending_ratio)
            x_observe = x_data[nr,subset]
            x_cov = est_subset_cov[np.ix_(subset,subset)]

            evals, evecs = np.linalg.eigh(x_cov)
            ev1 = evecs[:,np.argmax(evals)] # NOTE: PD matrix, must be the largest eigenvector
            # NOTE: The sign of the eigenvector will flip the predicted labels
            # NOTE: We record the similarity to the ground truth and flip them back in the end
            # FIXME: using oja's algorithm to update the eigenvectors with consistent sign
            rec_similarity_scalar = cosine_similarity(ev1.reshape(1,-1),true_mean_diff[subset].reshape(1,-1))[0,0]
            record_sign_eigv[nr] = np.sign(rec_similarity_scalar)
            y_pred = 1 if np.sum((x_observe - est_subset_means[subset]) * ev1) > 0 else 0
        else:
            subset = np.ones((num_views)).astype(bool)
            x_observe = x_data[nr]
            record_sign_eigv[nr] = 1.0 # default to positive
            y_pred = np.random.randint(2)
        record_predict[nr] = y_pred
        if remain_budget <=0:
            instant_cost = 0
            continue
        # algorithm update
        instant_cost = np.sum(costs[subset])
        remain_budget = max(0, remain_budget - instant_cost)
        # parameter update
        est_sample_means[subset] += x_observe
        est_sample_cov[np.ix_(subset,subset)] += np.outer(x_observe,x_observe)

        # Convert boolean subset to binary string, then to integer for indexing est_counts
        subset_int = int("".join(subset.astype(int).astype(str)), 2)
        est_counts[subset_int - 1] += 1 # OFFSET the index by 1
        raw_lambda = omd_lambda + step_size * (instant_cost - spending_ratio)
        omd_lambda = max(0, min(lambda_max, raw_lambda))

    # using the eigenvalue sign to flip the prediction for correct evaluation
    # Find indices where the eigenvector sign was flipped relative to the ground truth
    flipped_indices = np.where(record_sign_eigv < 0)[0]
    # Correct the predictions at these indices by flipping the label (0 -> 1, 1 -> 0)
    if len(flipped_indices) > 0:
        record_predict[flipped_indices] = 1 - record_predict[flipped_indices]

    # do a label matching here
    y_pred_mapped, _ = label_matching(y_labels,record_predict)
    acc_reward = np.sum(y_pred_mapped == y_labels)
    record_acc = np.cumsum(y_pred_mapped == y_labels) / np.arange(1,num_rounds+1)

    avg_acc = acc_reward / num_rounds
    return {"reward": acc_reward, "avg_acc":avg_acc, "record_acc":record_acc, "spending":budget - remain_budget}

## 10. Main Execution Block

This block orchestrates the entire simulation. It runs multiple trials, generates data, calculates optimal static performance, executes the dynamic simulation, and stores the results in a Pandas DataFrame, which is then saved to a CSV file.

In [14]:
def main():
    rng = np.random.default_rng(seed=seed)
    result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[],"optimal_reward":[]}
    for trial in range(num_trials):
        budget = num_rounds * budget_ratio
        costs = rng.random(size=(num_views,))
        # create free view (first view has zero cost)
        costs[0] = 0
        # normalize costs to sum to 1
        costs = costs/np.sum(costs)
        t_data, t_means, t_labels = generate_data(rng,num_rounds,2,num_views,seed)
        opt_dict = optimal_static(t_means,costs,budget,num_rounds)
        sim_dict = simulation(num_views,costs,num_rounds,budget,t_data,t_labels,**{"means":t_means})
        # accumulating results
        result_pd['trial'].append(trial)
        result_pd['reward'].append(sim_dict['reward'])
        result_pd['spending'].append(sim_dict['spending'])
        result_pd['avg_acc'].append(sim_dict['avg_acc'])
        result_pd['optimal_reward'].append(opt_dict['reward'])
    res_df = pd.DataFrame.from_dict(result_pd)
    res_df.to_csv(output_filename,index=False)
    print(f"Simulation results saved to {output_filename}")
    display(res_df.head())

# Run the main simulation
main()

Rounds:2000, Budget:1400.0, Optimal Reward: 1.9644e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9968e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9209e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9797e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9132e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9477e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9697e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9451e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9313e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.8680e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9837e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, Optimal Reward: 1.9704e+03, Optimal_cost: 1.4000e+03
Rounds:2000, Budget:1400.0, 

,trial,reward,spending,avg_acc,optimal_reward
0,0,1749,1006.825638,0.8745,1964.408315
1,1,1934,1030.715372,0.9670,1996.775601
2,2,1750,1342.021952,0.8750,1920.885499
3,3,1860,1093.289320,0.9300,1979.720988
4,4,1726,1225.151247,0.8630,1913.217837
